# Notebook 1 — Cargar Datos

**Proyecto Integrador - Módulo 5 - Henry**
**Pipeline MLOps - Predicción de Pago a Tiempo**
**Versión:** V1.0.1 (Avance 1, parte 1 de 2)

---

## Contexto y propósito

Este notebook es el **primer eslabón del pipeline MLOps**: toma los datos crudos, los valida, los castea a sus tipos correctos y los persiste en un formato eficiente para el resto del pipeline.

En un entorno de producción real, los datos ya estarían **materializados en una tabla del Data Warehouse o Datalake corporativo** como resultado de un proceso ETL upstream. Sin embargo, en este proyecto trabajamos con un dataset de ejemplo no productivo (`Base_de_datos.csv`). Por eso necesitamos este notebook intermedio que **simula** la capa de extracción: lee del CSV, valida, transforma tipos y persiste un parquet con metadatos preservados para que los notebooks posteriores no tengan que repetir esta lógica.

### Decisión arquitectónica explicada

Elegimos persistir en **Parquet** (no CSV) por tres razones:
1. **Preserva los tipos de datos** (CSV es solo texto, pierde la distinción entre `int`, `float`, `category`, `datetime`).
2. **Compresión columnar** ~5x más pequeña que CSV.
3. **Lectura ~10x más rápida** para análisis posterior.

Esta decisión refleja una práctica MLOps real: separar "datos crudos" de "datos curados" en formatos optimizados para cada uso.

---

## 1. Imports y configuración

In [1]:
# Librerías estándar
import json
import sys
from pathlib import Path

# Manipulación de datos
import pandas as pd
import numpy as np

# Visualización (chequeos rápidos)
import matplotlib.pyplot as plt

# Mostrar más columnas en las tablas para inspeccionar
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}' if abs(x) > 1 else f'{x:.4f}')

print(f"Python: {sys.version.split()[0]}")
print(f"pandas: {pd.__version__}")
print(f"numpy:  {np.__version__}")

Python: 3.10.12
pandas: 2.3.3
numpy:  2.2.6


## 2. Configuración de rutas (vía config.json)

Centralizamos las rutas y metadatos del proyecto en `config.json`. Esto evita hardcodear rutas en cada notebook y permite cambiar el dataset de origen sin tocar código.

**Decisión:** la ruta del dataset es relativa desde el notebook (`../../Base_de_datos.csv`) porque el notebook vive en `mlops_pipeline/src/` y el dataset en la raíz del repo.

In [2]:
# Cargar configuración
CONFIG_PATH = Path('config.json')
with open(CONFIG_PATH, 'r', encoding='utf-8') as f:
    config = json.load(f)

DATA_SOURCE = Path(config['data_source'])         # ../../Base_de_datos.csv
TARGET_VAR  = config['target_variable']           # Pago_atiempo
PROJECT     = config['project_code']              # mlops_credito_m5

# Carpeta de salida para datasets procesados
OUTPUT_DIR = Path('../../data_processed')
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Proyecto:        {PROJECT}")
print(f"Variable target: {TARGET_VAR}")
print(f"Fuente datos:    {DATA_SOURCE.resolve()}")
print(f"Carpeta salida:  {OUTPUT_DIR.resolve()}")

Proyecto:        mlops_credito_m5
Variable target: Pago_atiempo
Fuente datos:    /sessions/serene-amazing-davinci/mnt/PI/Base_de_datos.csv
Carpeta salida:  /sessions/serene-amazing-davinci/mnt/PI/data_processed


## 3. Lectura del dataset crudo

Leemos el CSV con `pd.read_csv`. Notar que pandas va a **inferir tipos automáticamente** — esa inferencia puede ser correcta o no. Nuestro trabajo en las próximas celdas es validar y corregir esa inferencia.



In [3]:
df = pd.read_csv(DATA_SOURCE)

print(f"Forma del dataset: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"Tamaño en memoria: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print()
print("Primeras 3 filas:")
df.head(3)

Forma del dataset: 10,763 filas × 23 columnas
Tamaño en memoria: 3.69 MB

Primeras 3 filas:


,tipo_credito,fecha_prestamo,capital_prestado,plazo_meses,edad_cliente,tipo_laboral,salario_cliente,total_otros_prestamos,cuota_pactada,puntaje,puntaje_datacredito,cant_creditosvigentes,huella_consulta,saldo_mora,saldo_total,saldo_principal,saldo_mora_codeudor,creditos_sectorFinanciero,creditos_sectorCooperativo,creditos_sectorReal,promedio_ingresos_datacredito,tendencia_ingresos,Pago_atiempo
0,7,2024-12-21 11:31:35,"3,692,160.00",10,42,Independiente,8000000,2500000,341296,88.77,695.00,10,5,0.0000,"51,258.00","51,258.00",0.0000,5,0,0,"908,526.00",Estable,1
1,4,2025-04-22 09:47:35,"840,000.00",6,60,Empleado,3000000,2000000,124876,95.23,789.00,3,1,0.0000,"8,673.00","8,673.00",0.0000,0,0,2,"939,017.00",Creciente,1
2,9,2026-01-08 12:22:40,"5,974,028.40",10,36,Independiente,4036000,829000,529554,47.61,740.00,4,5,0.0000,"18,702.00","18,702.00",0.0000,3,0,0,NaN,NaN,0


## 4. Inspección inicial: tipos inferidos por pandas

In [4]:
df.info(verbose=True, show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10763 entries, 0 to 10762
Data columns (total 23 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   tipo_credito                   10763 non-null  int64  
 1   fecha_prestamo                 10763 non-null  object 
 2   capital_prestado               10763 non-null  float64
 3   plazo_meses                    10763 non-null  int64  
 4   edad_cliente                   10763 non-null  int64  
 5   tipo_laboral                   10763 non-null  object 
 6   salario_cliente                10763 non-null  int64  
 7   total_otros_prestamos          10763 non-null  int64  
 8   cuota_pactada                  10763 non-null  int64  
 9   puntaje                        10763 non-null  float64
 10  puntaje_datacredito            10757 non-null  float64
 11  cant_creditosvigentes          10763 non-null  int64  
 12  huella_consulta                10763 non-null 

### Análisis de la inferencia

[Observado] pandas infiere tipos razonables pero hay decisiones que debemos revisar manualmente:

- `fecha_prestamo` se lee como **object** (string), no como datetime → hay que castear.
- `tipo_credito` se lee como **int64** pero conceptualmente es una **categoría nominal** codificada (los números no tienen orden).
- `tipo_laboral` y `tendencia_ingresos` son **object** (strings de texto) → categóricas.
- `Pago_atiempo` es **int64** pero es binario {0, 1} → más eficiente y semánticamente claro como `int8` o `category`.
- Las variables de saldo (`saldo_mora`, `saldo_total`, etc.) están como **float64** → OK si vienen limpias.

**Decisión:** vamos a redefinir todos los tipos explícitamente en la sección 6.

## 5. Validación de schema

Antes de transformar, validamos que el dataset tenga **todas y solo las columnas esperadas**. Si faltara una columna o sobrara, fallamos rápido (fail fast) — esto evita errores misteriosos río abajo.

In [5]:
# Schema esperado: las 23 columnas del dataset según la consigna
COLUMNAS_ESPERADAS = [
    'tipo_credito', 'fecha_prestamo', 'capital_prestado', 'plazo_meses',
    'edad_cliente', 'tipo_laboral', 'salario_cliente', 'total_otros_prestamos',
    'cuota_pactada', 'puntaje', 'puntaje_datacredito', 'cant_creditosvigentes',
    'huella_consulta', 'saldo_mora', 'saldo_total', 'saldo_principal',
    'saldo_mora_codeudor', 'creditos_sectorFinanciero', 'creditos_sectorCooperativo',
    'creditos_sectorReal', 'promedio_ingresos_datacredito', 'tendencia_ingresos',
    'Pago_atiempo'
]

faltantes = set(COLUMNAS_ESPERADAS) - set(df.columns)
sobrantes = set(df.columns) - set(COLUMNAS_ESPERADAS)

if faltantes:
    raise ValueError(f"ERROR: faltan columnas obligatorias: {faltantes}")
if sobrantes:
    print(f"AVISO: columnas extras (se ignoran): {sobrantes}")

print(f"Schema validado: {len(df.columns)}/{len(COLUMNAS_ESPERADAS)} columnas esperadas OK")
print(f"Target presente: {TARGET_VAR in df.columns}")

Schema validado: 23/23 columnas esperadas OK
Target presente: True


## 6. Caracterización de variables 

Categorizamos cada variable según su naturaleza estadística. Cada tipo necesita una transformación distinta (numéricas → escalar; categóricas nominales → one-hot; ordinales → ordinal encoding).

In [6]:
# Diccionario de caracterización - lo usaremos como contrato del proyecto
TIPOS_VARIABLES = {
    # NUMÉRICAS CONTINUAS (tienen sentido ordinal y decimales/escala continua)
    'numericas_continuas': [
        'capital_prestado', 'salario_cliente', 'total_otros_prestamos',
        'cuota_pactada', 'puntaje', 'puntaje_datacredito',
        'saldo_mora', 'saldo_total', 'saldo_principal', 'saldo_mora_codeudor',
        'promedio_ingresos_datacredito'
    ],
    # NUMÉRICAS DISCRETAS (enteros con sentido de magnitud)
    'numericas_discretas': [
        'plazo_meses', 'edad_cliente', 'cant_creditosvigentes', 'huella_consulta',
        'creditos_sectorFinanciero', 'creditos_sectorCooperativo', 'creditos_sectorReal'
    ],
    # CATEGÓRICAS NOMINALES (sin orden inherente)
    'categoricas_nominales': [
        'tipo_credito',       # códigos numéricos sin orden
        'tipo_laboral'        # Empleado, Independiente, etc.
    ],
    # CATEGÓRICAS ORDINALES (tienen orden natural)
    'categoricas_ordinales': [
        'tendencia_ingresos'  # Decreciente < Estable < Creciente
    ],
    # TEMPORALES
    'temporales': [
        'fecha_prestamo'
    ],
    # TARGET (dicotómica: binaria)
    'target': [
        'Pago_atiempo'        # 0/1
    ]
}

# Verificar que todas las variables están clasificadas (ninguna olvidada)
todas_clasificadas = set()
for lista in TIPOS_VARIABLES.values():
    todas_clasificadas.update(lista)

sin_clasificar = set(df.columns) - todas_clasificadas
clasificadas_inexistentes = todas_clasificadas - set(df.columns)

assert not sin_clasificar, f"Variables sin clasificar: {sin_clasificar}"
assert not clasificadas_inexistentes, f"Clasificadas pero no en el df: {clasificadas_inexistentes}"

print("Caracterización de las 23 variables:")
print("=" * 60)
for tipo, vars_ in TIPOS_VARIABLES.items():
    print(f"\n  {tipo} ({len(vars_)}):")
    for v in vars_:
        print(f"    • {v}")
print()
print("✓ Las 23 variables están clasificadas correctamente.")

Caracterización de las 23 variables:

  numericas_continuas (11):
    • capital_prestado
    • salario_cliente
    • total_otros_prestamos
    • cuota_pactada
    • puntaje
    • puntaje_datacredito
    • saldo_mora
    • saldo_total
    • saldo_principal
    • saldo_mora_codeudor
    • promedio_ingresos_datacredito

  numericas_discretas (7):
    • plazo_meses
    • edad_cliente
    • cant_creditosvigentes
    • huella_consulta
    • creditos_sectorFinanciero
    • creditos_sectorCooperativo
    • creditos_sectorReal

  categoricas_nominales (2):
    • tipo_credito
    • tipo_laboral

  categoricas_ordinales (1):
    • tendencia_ingresos

  temporales (1):
    • fecha_prestamo

  target (1):
    • Pago_atiempo

✓ Las 23 variables están clasificadas correctamente.


## 7. Casteo de tipos

Aplicamos los tipos correctos basados en la caracterización. Usamos `pd.to_numeric(errors='coerce')` para los numéricos por defensa: si alguna fila tuviera basura (e.g. una letra) la convierte en NaN en lugar de fallar.

In [7]:
# Backup pre-cast para auditar después
df_pre_cast = df.copy()

# 7.1 Castear temporales
df['fecha_prestamo'] = pd.to_datetime(df['fecha_prestamo'], errors='coerce')

# 7.2 Castear numéricos continuos
for col in TIPOS_VARIABLES['numericas_continuas']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# 7.3 Castear numéricos discretos (Int64 nullable permite NaN en columnas enteras)
for col in TIPOS_VARIABLES['numericas_discretas']:
    df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')

# 7.4 Castear categóricas nominales
for col in TIPOS_VARIABLES['categoricas_nominales']:
    df[col] = df[col].astype('category')

# 7.5 Castear categóricas ordinales (con orden explícito)
orden_tendencia = ['Decreciente', 'Estable', 'Creciente']
df['tendencia_ingresos'] = pd.Categorical(
    df['tendencia_ingresos'],
    categories=orden_tendencia,
    ordered=True
)

# 7.6 Castear target (más pequeño y semánticamente claro como int8)
df[TARGET_VAR] = df[TARGET_VAR].astype('int8')

print("Tipos post-casteo:")
print(df.dtypes.to_string())

Tipos post-casteo:
tipo_credito                           category
fecha_prestamo                   datetime64[ns]
capital_prestado                        float64
plazo_meses                               Int64
edad_cliente                              Int64
tipo_laboral                           category
salario_cliente                           int64
total_otros_prestamos                     int64
cuota_pactada                             int64
puntaje                                 float64
puntaje_datacredito                     float64
cant_creditosvigentes                     Int64
huella_consulta                           Int64
saldo_mora                              float64
saldo_total                             float64
saldo_principal                         float64
saldo_mora_codeudor                     float64
creditos_sectorFinanciero                 Int64
creditos_sectorCooperativo                Int64
creditos_sectorReal                       Int64
promedio_ingresos_dat

## 8. Auditoría del casteo: ¿perdimos información?

Cuando se castea con `coerce`, los valores no convertibles se vuelven NaN. Es **obligatorio auditar cuántos NaN introdujo el casteo** para detectar problemas de calidad de datos.

In [8]:
# Comparar nulos antes y después del cast (solo columnas que cambiaron de tipo)
auditoria = pd.DataFrame({
    'nulos_antes': df_pre_cast.isnull().sum(),
    'nulos_despues': df.isnull().sum(),
    'tipo_antes': df_pre_cast.dtypes.astype(str),
    'tipo_despues': df.dtypes.astype(str)
})
auditoria['nulos_introducidos'] = auditoria['nulos_despues'] - auditoria['nulos_antes']

# Mostrar solo columnas que tuvieron cambios
cambios = auditoria[
    (auditoria['nulos_introducidos'] != 0) |
    (auditoria['tipo_antes'] != auditoria['tipo_despues'])
]
print("Auditoría del casteo:")
print(cambios.to_string())

Auditoría del casteo:
                            nulos_antes  nulos_despues tipo_antes    tipo_despues  nulos_introducidos
tipo_credito                          0              0      int64        category                   0
fecha_prestamo                        0              0     object  datetime64[ns]                   0
plazo_meses                           0              0      int64           Int64                   0
edad_cliente                          0              0      int64           Int64                   0
tipo_laboral                          0              0     object        category                   0
cant_creditosvigentes                 0              0      int64           Int64                   0
huella_consulta                       0              0      int64           Int64                   0
creditos_sectorFinanciero             0              0      int64           Int64                   0
creditos_sectorCooperativo            0              0      

[Observado] El casteo cambió todos los tipos como esperábamos.

[Observado] **No se introdujeron nulos nuevos** por errores de conversión. Los nulos que ven (en `promedio_ingresos_datacredito` y `tendencia_ingresos`) ya estaban en el dataset original y son **información**, no error.

## 9. Análisis de nulos: la sociología del dato faltante

Los nulos en `promedio_ingresos_datacredito` y `tendencia_ingresos` NO son ruido aleatorio. Son **información sobre el cliente**: clientes sin historial en datacrédito (probablemente nuevos en el sistema financiero formal).

> En sociología económica esto se llama "informalidad financiera", segmentos de población que no aparecen en burós de crédito porque nunca tuvieron productos formales. Tratar este nulo como dato perdido (imputar con la media) **destruiría una variable explicativa**. La estrategia correcta es marcarlo como categoría aparte ("Sin_Historial").

Vamos a verificar esta hipótesis: si los nulos en las dos variables aparecen en las MISMAS filas, son el mismo fenómeno.

In [9]:
nulos_por_var = df.isnull().sum()
print("Nulos por variable:")
print(nulos_por_var[nulos_por_var > 0].to_string())
print()

# Test: ¿los nulos están en las mismas filas?
nulos_promedio = df['promedio_ingresos_datacredito'].isnull()
nulos_tendencia = df['tendencia_ingresos'].isnull()

interseccion = (nulos_promedio & nulos_tendencia).sum()
solo_promedio = (nulos_promedio & ~nulos_tendencia).sum()
solo_tendencia = (~nulos_promedio & nulos_tendencia).sum()

print(f"Filas con NULL en ambas:           {interseccion:,}")
print(f"Filas con NULL solo en promedio:   {solo_promedio:,}")
print(f"Filas con NULL solo en tendencia:  {solo_tendencia:,}")
print()
print(f"% del dataset con nulos correlacionados: {interseccion/len(df)*100:.2f}%")

Nulos por variable:
puntaje_datacredito                 6
saldo_mora                        156
saldo_total                       156
saldo_principal                   405
saldo_mora_codeudor               590
promedio_ingresos_datacredito    2930
tendencia_ingresos               2990

Filas con NULL en ambas:           2,930
Filas con NULL solo en promedio:   0
Filas con NULL solo en tendencia:  60

% del dataset con nulos correlacionados: 27.22%


Confirmado: **Los nulos en ambas variables son las MISMAS filas.** Esto valida la hipótesis: hay un subgrupo de clientes (~14% del dataset) sin historial en datacrédito.



## 10. Unificación de representación de nulos

Verificamos que no haya múltiples formas de representar valores nulos en las columnas categóricas (strings tipo `"NaN"`, `"NA"`, `"null"`, `""`, `"-"`).

In [10]:
# Revisar valores únicos en variables tipo string/categórica para detectar nulos disfrazados
patrones_nulos = ['nan', 'NaN', 'NA', 'N/A', 'null', 'Null', 'NULL', '', ' ', '-', '?']

print("Búsqueda de nulos disfrazados en variables categóricas:")
print("=" * 60)
for col in TIPOS_VARIABLES['categoricas_nominales'] + TIPOS_VARIABLES['categoricas_ordinales']:
    valores_unicos = df[col].dropna().astype(str).unique()
    sospechosos = [v for v in valores_unicos if v.strip() in patrones_nulos]
    print(f"\n  {col}:")
    print(f"    Categorías ({len(valores_unicos)}): {sorted(valores_unicos)[:10]}")
    if sospechosos:
        print(f"    ⚠ Sospechosos de nulos: {sospechosos}")
    else:
        print(f"    ✓ Sin nulos disfrazados")

Búsqueda de nulos disfrazados en variables categóricas:

  tipo_credito:
    Categorías (6): ['10', '4', '6', '68', '7', '9']
    ✓ Sin nulos disfrazados

  tipo_laboral:
    Categorías (2): ['Empleado', 'Independiente']
    ✓ Sin nulos disfrazados

  tendencia_ingresos:
    Categorías (3): ['Creciente', 'Decreciente', 'Estable']
    ✓ Sin nulos disfrazados


Conclusión: No hay nulos disfrazados — la calidad de los datos categóricos es buena.

## 11. Eliminación de variables irrelevantes

Antes de eliminar nada, evaluamos cada variable contra el criterio: **¿esta variable podría ayudar a predecir el target?**

Posibles candidatas a eliminar:
- Variables constantes (un solo valor)
- Variables casi constantes (>99% un mismo valor)
- IDs únicos (alta cardinalidad sin sentido predictivo)

Vamos a chequear sistemáticamente.

In [11]:
print("Análisis de variabilidad por columna:")
print("=" * 70)
for col in df.columns:
    n_unicos = df[col].nunique(dropna=False)
    pct_unicos = n_unicos / len(df) * 100
    valor_mas_frecuente_pct = (df[col].value_counts(dropna=False, normalize=True).iloc[0] * 100) if n_unicos > 0 else 0

    flag = ''
    if n_unicos == 1:
        flag = '⚠ CONSTANTE'
    elif valor_mas_frecuente_pct > 99:
        flag = '⚠ CASI CONSTANTE'
    elif pct_unicos > 95 and col != 'fecha_prestamo':
        flag = '⚠ CASI ID'

    print(f"  {col:35s}  uniq={n_unicos:6d}  modo={valor_mas_frecuente_pct:5.1f}%  {flag}")

Análisis de variabilidad por columna:
  tipo_credito                         uniq=     6  modo= 72.0%  
  fecha_prestamo                       uniq= 10758  modo=  0.0%  
  capital_prestado                     uniq=  7306  modo=  1.2%  
  plazo_meses                          uniq=    18  modo= 30.9%  
  edad_cliente                         uniq=    54  modo=  3.2%  
  tipo_laboral                         uniq=     2  modo= 62.8%  
  salario_cliente                      uniq=  1385  modo=  7.7%  
  total_otros_prestamos                uniq=  1538  modo= 10.3%  
  cuota_pactada                        uniq=  9836  modo=  0.1%  
  puntaje                              uniq=   248  modo= 87.4%  
  puntaje_datacredito                  uniq=   316  modo=  1.3%  
  cant_creditosvigentes                uniq=    39  modo= 13.0%  
  huella_consulta                      uniq=    28  modo= 18.3%  
  saldo_mora                           uniq=    56  modo= 98.0%  
  saldo_total                         

[Observado] **No hay variables a eliminar en esta etapa.** Todas las variables tienen variabilidad razonable y aportan información potencialmente útil.

`fecha_prestamo` aparece con muchas categorías únicas pero es esperable (timestamps casi únicos por cliente). No la eliminamos.

## 12. Distribución del target: detección de desbalance

In [12]:
distribucion = df[TARGET_VAR].value_counts(dropna=False).sort_index()
distribucion_pct = df[TARGET_VAR].value_counts(dropna=False, normalize=True).sort_index() * 100

print(f"Distribución de '{TARGET_VAR}':")
print("=" * 50)
for clase, count in distribucion.items():
    nombre = 'Impago (0)' if clase == 0 else 'Pago a tiempo (1)'
    pct = distribucion_pct.loc[clase]
    print(f"  Clase {clase} ({nombre:20s}): {count:>7,} ({pct:5.2f}%)")

ratio = distribucion[1] / distribucion[0]
print(f"\nRatio de desbalance: 1:{ratio:.1f} (por cada cliente que no paga, hay ~{ratio:.0f} que sí pagan)")

Distribución de 'Pago_atiempo':
  Clase 0 (Impago (0)          ):     511 ( 4.75%)
  Clase 1 (Pago a tiempo (1)   ):  10,252 (95.25%)

Ratio de desbalance: 1:20.1 (por cada cliente que no paga, hay ~20 que sí pagan)


Conclusión: **Desbalance severo: 95.2% / 4.8%.**

 **Técnicas necesarias**:
   - `stratify=y` en el split train/test para preservar el desbalance en ambos conjuntos.
   - `class_weight='balanced'` en los modelos que lo soporten.
   - Considerar SMOTE u oversampling de la clase minoritaria.



---

## 13. Persistencia del dataset limpio

Guardamos el dataset transformado en formato **Parquet** (preserva tipos, comprimido) en `data_processed/`. Este archivo es el **contrato de salida** de este notebook: lo consume `comprension_eda.ipynb` y los notebooks/scripts posteriores.

In [13]:
OUTPUT_PATH = OUTPUT_DIR / 'dataset_limpio.parquet'
df.to_parquet(OUTPUT_PATH, index=False, compression='snappy')

# Verificación: releer y comparar
df_check = pd.read_parquet(OUTPUT_PATH)
assert df_check.shape == df.shape, "Forma diferente al releer!"
assert list(df_check.columns) == list(df.columns), "Columnas en orden distinto!"

# Reportar diferencias de tipos (parquet no preserva category con códigos enteros)
def familia_tipo(dt):
    s = str(dt)
    if 'datetime' in s: return 'datetime'
    if 'category' in s: return 'category'
    if 'Int' in s or 'int' in s: return 'integer'
    if 'float' in s: return 'float'
    return s

tipos_orig = df.dtypes.apply(familia_tipo)
tipos_check = df_check.dtypes.apply(familia_tipo)
diferencias = tipos_orig[tipos_orig != tipos_check]
if not diferencias.empty:
    print("Nota: parquet no preserva exactamente algunas conversiones a 'category' con códigos enteros.")
    print("Las columnas afectadas se vuelven a castear en comprension_eda.ipynb.")
    print("Diferencias detectadas (esperadas):")
    for col, _ in diferencias.items():
        print(f"  • {col}: {df[col].dtype} → {df_check[col].dtype}")

tamano_csv = DATA_SOURCE.stat().st_size / 1024**2
tamano_parquet = OUTPUT_PATH.stat().st_size / 1024**2

print(f"✓ Dataset persistido en: {OUTPUT_PATH.resolve()}")
print(f"  Filas:    {df_check.shape[0]:,}")
print(f"  Columnas: {df_check.shape[1]}")
print(f"\nNota: Parquet preserva familias de tipo (category, datetime, int, float).")
print(f"El orden categórico de tendencia_ingresos se reaplica al cargar (ver siguiente notebook).")
print(f"\nComparación de tamaños:")
print(f"  CSV original: {tamano_csv:.2f} MB")
print(f"  Parquet:      {tamano_parquet:.2f} MB  ({tamano_csv/tamano_parquet:.1f}× más chico)")

Nota: parquet no preserva exactamente algunas conversiones a 'category' con códigos enteros.
Las columnas afectadas se vuelven a castear en comprension_eda.ipynb.
Diferencias detectadas (esperadas):
  • tipo_credito: category → int64
✓ Dataset persistido en: /sessions/serene-amazing-davinci/mnt/PI/data_processed/dataset_limpio.parquet
  Filas:    10,763
  Columnas: 23

Nota: Parquet preserva familias de tipo (category, datetime, int, float).
El orden categórico de tendencia_ingresos se reaplica al cargar (ver siguiente notebook).

Comparación de tamaños:
  CSV original: 1.40 MB
  Parquet:      0.51 MB  (2.7× más chico)


## 14. Resumen final del notebook

### Lo que hicimos
1. ✓ Validamos el schema (23 columnas esperadas presentes).
2. ✓ Caracterizamos las variables: 11 numéricas continuas, 7 numéricas discretas, 2 categóricas nominales, 1 ordinal, 1 temporal, 1 target.
3. ✓ Casteamos tipos correctamente (datetime, Int64, category con orden cuando aplica).
4. ✓ Auditamos el casteo: no introdujimos nulos nuevos.
5. ✓ Confirmamos que los nulos en `promedio_ingresos_datacredito` y `tendencia_ingresos` son **información** (clientes sin historial), no error.
6. ✓ Verificamos que no hay nulos disfrazados ni variables a eliminar.
7. ✓ Detectamos desbalance severo del target (95% / 5%) — impactará el modelado.
8. ✓ Persistimos en Parquet para uso río abajo.
